> **File đính kèm của [`BAO_CAO_TANG_METRIC_S.md`](BAO_CAO_TANG_METRIC_S.md)**.

**Phụ thuộc (đều đã có trong notebook chính):** `sample_frames_stamped`, `vlm_generate_oom_safe`,
`_extract_json`, `_safe_float`, `_clip01`, `VLM_CFG`, `GROUNDING_PROMPT`, `spatial_score`,
`score_predictions`, `accident_score`, `diverse_videos`, `diverse_labels_df`, `labels_clean`,
`CONST`, `SIGMA_X`, `SIGMA_Y`, `COLLISION_TYPES`, `SEED`, `OUTPUT_DIR`, `SCENE_BY_PATH`,
`classify_type_cascade`, `apply_scene_type_postfix`, `stage1_full_scan` (NumPro, cell 109),
`stage2_time_refine_numbered` (cell 111), `PARSE_FAIL_STREAK`, `PARSE_FAIL_ABORT`,
`real_videos`, `const_eval`, `VLM_AVAILABLE`.

## Bước 0 — Cache Stage 1 + 2 (chạy MỘT LẦN, ~15 phút GPU)

Stage 1/2 dùng greedy decode (`do_sample=False`) nên kết quả **tất định** — cache lại `t_final`,
toạ độ thô và loại từ Stage 1 để mọi thí nghiệm Stage 3 phía sau chỉ tốn ~3 phút thay vì ~15 phút.
Đồng thời gắn ground truth vào từng dòng để đo được cả `S_ORACLE` (grounding tại thời điểm THẬT).

In [1]:
import pathlib, time

CACHE_S12 = OUTPUT_DIR / 'rows_stage12_numpro.csv'


def _duration_of(vp):
    cap = cv2.VideoCapture(str(vp))
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return (n / fps) if fps > 0 else 20.0


if CACHE_S12.exists():
    rows_s12 = pd.read_csv(CACHE_S12).to_dict('records')
    print(f'[CACHE] doc lai {len(rows_s12)} dong tu {CACHE_S12}')
else:
    rows_s12, _t0 = [], time.time()
    for _i, vp in enumerate(diverse_videos, 1):
        duration = _duration_of(vp)
        scene = SCENE_BY_PATH.get('videos/' + vp.name)
        pred = stage1_full_scan(vp, duration, scene)
        t_final = stage2_time_refine_numbered(vp, pred['accident_time'], duration)
        rows_s12.append({
            'path': str(vp), 'duration': duration, 't_final': t_final,
            't_stage1': pred['accident_time'],
            'x_stage1': pred['center_x'], 'y_stage1': pred['center_y'],
            'type_stage1': pred['type'],
        })
        print(f"[{_i:2d}/{len(diverse_videos)}] {vp.name:42s} "
              f"t_final={t_final:6.2f}  xy1=({pred['center_x']:.2f},{pred['center_y']:.2f})  "
              f"({time.time()-_t0:5.0f}s)")
    pd.DataFrame(rows_s12).to_csv(CACHE_S12, index=False)
    print(f'[CACHE] da luu {CACHE_S12}')


_gt = diverse_labels_df.copy()
_gt['stem'] = _gt['rgb_path'].map(lambda p: pathlib.Path(p).stem)
_gt = _gt.set_index('stem')
for r in rows_s12:
    _st = pathlib.Path(r['path']).stem
    r['t_gt'] = float(_gt.loc[_st, 'accident_time'])
    r['x_gt'] = float(_gt.loc[_st, 'center_x'])
    r['y_gt'] = float(_gt.loc[_st, 'center_y'])
    r['type_gt'] = str(_gt.loc[_st, 'type'])

print(f"[STATUS] cache san sang: {len(rows_s12)} video | "
      f"|t_final - t_gt| trung binh = "
      f"{np.mean([abs(r['t_final'] - r['t_gt']) for r in rows_s12]):.2f}s")



[ 1/20] Town03_head-on_wet_48.mp4                  t_final=  2.91  xy1=(0.52,0.44)  (   43s)
[ 2/20] Town06_head-on_wet_06.mp4                  t_final= 15.53  xy1=(0.66,0.31)  (   95s)
[ 3/20] Town03_head-on_night_40.mp4                t_final=  6.03  xy1=(0.34,0.52)  (  142s)
[ 4/20] Town06_head-on_wet_01.mp4                  t_final=  9.47  xy1=(0.49,0.58)  (  188s)
[ 5/20] Town04_rear-end_sunset_13.mp4              t_final=  4.03  xy1=(0.37,0.55)  (  236s)
[ 6/20] Town04_rear-end_rain_09.mp4                t_final=  6.47  xy1=(0.58,0.49)  (  281s)
[ 7/20] Town05_rear-end_rain_142.mp4               t_final= 12.94  xy1=(0.42,0.47)  (  330s)
[ 8/20] Town07_rear-end_rain_28.mp4                t_final=  8.82  xy1=(0.28,0.51)  (  377s)
[ 9/20] Town05_sideswipe_clear_04.mp4              t_final=  5.44  xy1=(0.53,0.74)  (  425s)
[10/20] Town04_sideswipe_wet_10.mp4                t_final=  9.97  xy1=(0.44,0.33)  (  470s)
[11/20] Town06_sideswipe_wet_06.mp4                t_final= 11.53  xy

## Bước 1 — Harness đo S + baseline mới (việc số 0 của kế hoạch)

`S = 0.3987` cũ được đo trên `t_final` sai trung bình **3.10 s**; sau nâng cấp NumPro (cell
109/111) sai số còn **2.15 s** — nên phải **đo lại baseline** trước khi so sánh bất kỳ cải tiến
nào. Harness cũng đo `S_ORACLE` (grounding tại thời điểm THẬT) — chính là **trần** của Stage 3
hiện tại; khoảng cách `ORACLE − PIPELINE` là phần điểm còn phụ thuộc vào chất lượng T.

In [2]:
def eval_S(grounding_fn, name='', use_gt_time=False, rows=None,
           lam=(1.0, 1.0), bias=(0.0, 0.0), fallback='const',
           lam_lo=None, spread_gate=None, verbose=True):
    rows = rows_s12 if rows is None else rows
    mu_x, mu_y = CONST['center_x'], CONST['center_y']
    out = []
    for r in rows:
        vp = pathlib.Path(r['path'])
        t = float(r['t_gt'] if use_gt_time else r['t_final'])

        res = grounding_fn(vp, t, r)
        spread, used = np.nan, 'model'
        if isinstance(res, tuple) and len(res) == 3:
            res, spread = (res[0], res[1]), res[2]

        if res is None:
            used = fallback
            res = ((mu_x, mu_y) if fallback == 'const'
                   else (float(r['x_stage1']), float(r['y_stage1'])))
        lx, ly = lam
        if lam_lo is not None and spread_gate is not None and np.isfinite(spread) \
                and spread > spread_gate:
            lx, ly = lam_lo

        x = _clip01(mu_x + lx * (res[0] + bias[0] - mu_x))
        y = _clip01(mu_y + ly * (res[1] + bias[1] - mu_y))

        out.append({'stem': vp.stem, 'type_gt': r['type_gt'],
                    'x_raw': res[0], 'y_raw': res[1], 'spread': spread,
                    'x': x, 'y': y, 'used': used,
                    'S': spatial_score(x, y, r['x_gt'], r['y_gt']),
                    'dx': x - r['x_gt'], 'dy': y - r['y_gt']})
    df = pd.DataFrame(out)
    _rng = np.random.default_rng(0)
    boot = [df['S'].sample(len(df), replace=True,
                           random_state=int(_rng.integers(1e9))).mean()
            for _ in range(2000)]
    lo, hi = np.percentile(boot, [2.5, 97.5])

    if verbose:
        print(f"[S] {name or grounding_fn.__name__:34s} "
              f"{'ORACLE' if use_gt_time else 'PIPELINE'}  "
              f"S = {df['S'].mean():.4f}  (95% CI {lo:.4f}-{hi:.4f})  "
              f"fallback {int((df['used'] != 'model').sum())}/{len(df)}")
    return df


def compare_S(df_new, df_base, name='moi'):
    """Bang delta tung video -- cai thien la HE THONG hay chi may man o 1-2 video?
    Voi n=20, mot video tu 0.02 len 0.95 da day trung binh +0.046 ma khong chung
    minh duoc gi."""
    m = df_base[['stem', 'type_gt', 'S']].rename(columns={'S': 'S_base'}).merge(
        df_new[['stem', 'S']].rename(columns={'S': 'S_new'}), on='stem')
    m['delta'] = m['S_new'] - m['S_base']
    print(f"\n[{name}] S {df_base['S'].mean():.4f} -> {df_new['S'].mean():.4f}  "
          f"({df_new['S'].mean() - df_base['S'].mean():+.4f})  |  "
          f"tot len {int((m['delta'] > 0.01).sum())}/{len(m)} video, "
          f"te di {int((m['delta'] < -0.01).sum())}/{len(m)}")
    display(m.sort_values('delta').round(4))
    return m


def g_base(vp, t, row=None):
    return stage3_grounding(vp, t)


eval_base_pipe = eval_S(g_base, 'stage3_grounding (goc)')
eval_base_orac = eval_S(g_base, 'stage3_grounding (goc)', use_gt_time=True)

_const_S = np.mean([spatial_score(CONST['center_x'], CONST['center_y'],
                                  r['x_gt'], r['y_gt']) for r in rows_s12])
print(f'\n[FLOOR]  constant (0.51, 0.51)  S = {_const_S:.4f}')
print(f'[CU]     bao cao o cell 94        S = 0.3987  (do voi t_final CU)')
print(f'[TRAN]   S_ORACLE - S_PIPELINE    = '
      f"{eval_base_orac['S'].mean() - eval_base_pipe['S'].mean():+.4f}  "
      '<- phan diem con phu thuoc vao chat luong T')



[S] stage3_grounding (goc)             PIPELINE  S = 0.4421  (95% CI 0.3352-0.5518)  fallback 1/20
[S] stage3_grounding (goc)             ORACLE    S = 0.4987  (95% CI 0.3901-0.6055)  fallback 1/20

[FLOOR]  constant (0.51, 0.51)  S = 0.2159
[CU]     bao cao o cell 94        S = 0.3987  (do voi t_final CU)
[TRAN]   S_ORACLE - S_PIPELINE    = +0.0566  <- phan diem con phu thuoc vao chat luong T

## Bước 2 — Chẩn đoán trước khi sửa

Dump output thô của grounding + đếm toạ độ suy biến + đo bias/độ tán. **Số ở đây quyết định
nhánh nào được ưu tiên** — rất nhiều "cải tiến prompt" thực chất chỉ là sửa lỗi parse.

In [3]:
print('=' * 78)
for r in rows_s12[:8]:
    vp, t = pathlib.Path(r['path']), float(r['t_final'])
    frames = sample_frames_stamped(vp, 1.0, t, t + 1e-3, limit=1,
                                   max_side=VLM_CFG['grounding_max_side'], burn=False)
    raw = vlm_generate_oom_safe(GROUNDING_PROMPT, frames, 64, min_frames=1)
    print(f"{vp.stem:40s} t={t:6.2f} (gt {r['t_gt']:5.2f})")
    print(f"   RAW    : {raw!r}")
    print(f"   parsed : {_extract_json(raw)}")
    print(f"   gt xy  : ({r['x_gt']:.3f}, {r['y_gt']:.3f})")
print('=' * 78)

_d = eval_base_pipe
_n_zero = int(((_d['x_raw'].abs() < 0.02) & (_d['y_raw'].abs() < 0.02)).sum())
print(f"\n[DIAG] toa do (0,0)      : {_n_zero}/{len(_d)}")
print(f"[DIAG] fallback (None)   : {int((_d['used'] != 'model').sum())}/{len(_d)}")
print(f"[DIAG] gia tri x phan biet: {_d['x_raw'].round(3).nunique()}/{len(_d)}  "
      f"| y: {_d['y_raw'].round(3).nunique()}/{len(_d)}")

print(f"\n[DIAG] bias  mean(dx)={_d['dx'].mean():+.4f}  mean(dy)={_d['dy'].mean():+.4f}")
print(f"[DIAG] sai so std(dx)={_d['dx'].std():.4f}  std(dy)={_d['dy'].std():.4f}"
      f"   (so sanh: sigma_x={SIGMA_X:.4f}, sigma_y={SIGMA_Y:.4f})")

_tau_x = float(labels_clean['center_x'].std())
_tau_y = float(labels_clean['center_y'].std())
_ok = _d[(_d['x_raw'].abs() >= 0.02) | (_d['y_raw'].abs() >= 0.02)]
_vx, _vy = _ok['x_raw'].var(ddof=1), _ok['y_raw'].var(ddof=1)
print(f"\n[DIAG] std du doan  x={np.sqrt(_vx):.4f}  y={np.sqrt(_vy):.4f}")
print(f"[DIAG] std nhan     x={_tau_x:.4f}  y={_tau_y:.4f}")
print(f"[DIAG] lambda* theo moment = ({min(1.0, _tau_x**2/_vx):.3f}, "
      f"{min(1.0, _tau_y**2/_vy):.3f})   <- diem khoi dau cho shrinkage (Buoc 4)")



Town03_head-on_wet_48                    t=  2.91 (gt  3.50)
   RAW    : '{"point": [652, 447]}'
   parsed : {'point': [652, 447]}
   gt xy  : (0.598, 0.454)
Town06_head-on_wet_06                    t= 15.53 (gt 17.50)
   RAW    : '{"point": [703, 291]}'
   parsed : {'point': [703, 291]}
   gt xy  : (0.671, 0.335)
Town03_head-on_night_40                  t=  6.03 (gt  6.45)
   RAW    : '{"reason": "impact near the crossing"} {"point": [418, 566]}'
   parsed : None
   gt xy  : (0.402, 0.517)
Town06_head-on_wet_01                    t=  9.47 (gt  7.25)
   RAW    : '{"point": [488, 617]}'
   parsed : {'point': [488, 617]}
   gt xy  : (0.529, 0.566)
Town04_rear-end_sunset_13                t=  4.03 (gt  4.75)
   RAW    : '{"point": [352, 549]}'
   parsed : {'point': [352, 549]}
   gt xy  : (0.388, 0.522)
Town04_rear-end_rain_09                  t=  6.47 (gt  4.30)
   RAW    : '{"point": [572, 500]}'
   parsed : {'point': [572, 500]}
   gt xy  : (0.573, 0.481)
Town05_rear-end_rain_142     

## Bước 3 — v1: parser cứng hơn + chặn toạ độ suy biến (0 GPU thêm)

Sửa 3 lỗi parse có thật của bản gốc: (1) `_extract_json` dùng regex greedy nên thất bại khi
output có 2 khối JSON; (2) regex fallback bắt **mọi** cặp số trong ngoặc, kể cả trong câu văn —
một đường sinh ra `(0, 0)`; (3) quy đổi thang [0,1000] xét từng số riêng lẻ. Đồng thời chặn toạ
độ suy biến `(0,0)`/`(1,1)` (2/20 clip calibration, ≥1/5 clip thật đang dính, mỗi clip ăn S=0)
và kẹp kết quả vào dải phân vị 0.5–99.5% của nhãn.

In [4]:
def _extract_json_all(text):
    """Tra ve MOI dict JSON tim duoc (khong greedy nhu _extract_json goc)."""
    out = []
    for m in re.finditer(r'\{', text or ''):
        for end in re.finditer(r'\}', text[m.start():]):
            frag = text[m.start(): m.start() + end.end()]
            try:
                obj = json.loads(frag)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict):
                out.append(obj)
                break
    return out


XY_LO_X, XY_HI_X = (float(labels_clean['center_x'].quantile(0.005)),
                    float(labels_clean['center_x'].quantile(0.995)))
XY_LO_Y, XY_HI_Y = (float(labels_clean['center_y'].quantile(0.005)),
                    float(labels_clean['center_y'].quantile(0.995)))
print(f'[STATUS] dai toa do hop le: x[{XY_LO_X:.3f},{XY_HI_X:.3f}] '
      f'y[{XY_LO_Y:.3f},{XY_HI_Y:.3f}]')


def _to_unit(x, y):
    """Quy doi ve [0,1] dua tren DO LON CUA CA CAP, khong xet tung so rieng le."""
    if max(abs(x), abs(y)) > 1.5:
        scale = 1000.0 if max(abs(x), abs(y)) > 100.0 else 100.0
        return x / scale, y / scale
    return x, y


def parse_point(text):
    """Bat toa do tu output tho. Tra None khi khong chac -- 'khong biet' tot hon
    'doan (0,0)': caller se rot ve prior thay vi ve goc tren trai."""
    for j in _extract_json_all(text):
        cand = None
        for key in ('point', 'point_2d', 'center', 'impact_point'):
            v = j.get(key)
            if isinstance(v, (list, tuple)) and len(v) >= 2:
                cand = (_safe_float(v[0], np.nan), _safe_float(v[1], np.nan))
                break
        if cand is None and 'center_x' in j and 'center_y' in j:
            cand = (_safe_float(j['center_x'], np.nan), _safe_float(j['center_y'], np.nan))
        if cand is None:
            v = j.get('points')
            if isinstance(v, (list, tuple)) and v and isinstance(v[0], (list, tuple)):
                cand = (_safe_float(v[0][0], np.nan), _safe_float(v[0][1], np.nan))
        if cand and all(np.isfinite(c) for c in cand):
            return _to_unit(*cand)

    m = re.search(r'[\[\(]\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*[\]\)]', text or '')
    if m:
        return _to_unit(float(m.group(1)), float(m.group(2)))
    return None


def sanitize_point(pt):
    """None neu suy bien; nguoc lai kep vao dai phan vi nhan."""
    if pt is None:
        return None
    x, y = pt
    if not (np.isfinite(x) and np.isfinite(y)):
        return None
    if (abs(x) < 0.02 and abs(y) < 0.02) or (x > 0.98 and y > 0.98):
        return None
    return (float(np.clip(x, XY_LO_X, XY_HI_X)), float(np.clip(y, XY_LO_Y, XY_HI_Y)))


def g_v1(vp, t, row=None):
    """stage3_grounding voi parser cung hon + chan suy bien. Cung 1 lan goi."""
    frames = sample_frames_stamped(vp, 1.0, t, t + 1e-3, limit=1,
                                   max_side=VLM_CFG['grounding_max_side'], burn=False)
    if not frames:
        return None
    return sanitize_point(parse_point(
        vlm_generate_oom_safe(GROUNDING_PROMPT, frames, 64, min_frames=1)))


eval_v1 = eval_S(g_v1, 'v1: parser + chan suy bien')
compare_S(eval_v1, eval_base_pipe, 'v1')



[STATUS] dai toa do hop le: x[0.128,0.833] y[0.078,0.916]
[S] v1: parser + chan suy bien         PIPELINE  S = 0.4676  (95% CI 0.3618-0.5722)  fallback 2/20

[v1] S 0.4421 -> 0.4676  (+0.0255)  |  tot len 3/20 video, te di 0/20
                         stem   type_gt  S_base   S_new   delta
0       Town03_head-on_wet_48   head-on  0.7742  0.7742  0.0000
1       Town06_head-on_wet_06   head-on  0.5731  0.5731  0.0000
..                        ...       ...     ...     ...     ...
2     Town03_head-on_night_40   head-on  0.3105  0.4151  0.1046
6    Town05_rear-end_rain_142  rear-end  0.0000  0.1959  0.1959
12  Town10HD_single_sunset_05    single  0.0000  0.2103  0.2103

[20 rows x 5 columns]

## Bước 4 — Shrinkage λ về prior (0 GPU thêm, chỉ hậu xử lý)

Dự đoán đang **tán rộng hơn nhãn** (std x: 0.173 so với 0.130 sau khi bỏ suy biến) — với hàm
điểm Gaussian, co dự đoán về prior `CONST=(0.51, 0.51)` làm tăng điểm kỳ vọng (Wiener /
James–Stein, `λ* = τ²/(τ²+s²)`). λ đặt bằng **phương pháp moment** `λ* = Var(nhãn)/Var(dự đoán)`
— không cần nhãn từng video, nên áp lại được trên chính 2027 dự đoán của tập test, tránh overfit
vào 20 video CARLA. Sweep chỉ để đối chiếu.

In [5]:
def rescore(df, lam, bias=(0.0, 0.0)):
    mu_x, mu_y = CONST['center_x'], CONST['center_y']
    xs = np.clip(mu_x + lam[0] * (df['x'] + bias[0] - mu_x), 0, 1)
    ys = np.clip(mu_y + lam[1] * (df['y'] + bias[1] - mu_y), 0, 1)
    gt = pd.DataFrame(rows_s12)
    gt['stem'] = gt['path'].map(lambda p: pathlib.Path(p).stem)
    gt = gt.set_index('stem').loc[df['stem']]
    return float(np.mean([spatial_score(a, b, gx, gy) for a, b, gx, gy
                          in zip(xs, ys, gt['x_gt'], gt['y_gt'])]))


print('[SWEEP] lambda dong nhat 2 truc')
for _lam in (1.0, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.5):
    print(f'   lam={_lam:.2f}  S={rescore(eval_v1, (_lam, _lam)):.4f}')

print('\n[SWEEP] lambda rieng tung truc (sigma_y > sigma_x nen 2 truc khong doi xung)')
_best = (None, -1)
for _lx in (1.0, 0.9, 0.8, 0.7, 0.6, 0.5):
    _row = []
    for _ly in (1.0, 0.9, 0.8, 0.7, 0.6, 0.5):
        _s = rescore(eval_v1, (_lx, _ly))
        _row.append(f'{_s:.4f}')
        if _s > _best[1]:
            _best = ((_lx, _ly), _s)
    print(f'   lam_x={_lx:.1f} | ' + '  '.join(_row))
print(f'\n[BEST sweep] lam={_best[0]}  S={_best[1]:.4f}')

_ok = eval_v1[eval_v1['used'] == 'model']
_lam_mom = (min(1.0, float(labels_clean['center_x'].std())**2 / _ok['x_raw'].var(ddof=1)),
            min(1.0, float(labels_clean['center_y'].std())**2 / _ok['y_raw'].var(ddof=1)))
print(f'[MOMENT   ] lam=({_lam_mom[0]:.3f}, {_lam_mom[1]:.3f})  '
      f'S={rescore(eval_v1, _lam_mom):.4f}')

_bx, _by = eval_v1['dx'].mean(), eval_v1['dy'].mean()
_sx, _sy = eval_v1['dx'].sem(), eval_v1['dy'].sem()
_bias = (-_bx if abs(_bx) > _sx else 0.0, -_by if abs(_by) > _sy else 0.0)
print(f'[BIAS     ] mean(dx)={_bx:+.4f}+-{_sx:.4f}  mean(dy)={_by:+.4f}+-{_sy:.4f}'
      f'  -> ap dung ({_bias[0]:.1f}, {_bias[1]:.1f})')
print(f'[BIAS     ] S={rescore(eval_v1, _lam_mom, _bias):.4f}')

print(f'[QUYET DINH] sweep ({_best[1]:.4f}) vs moment ({rescore(eval_v1, _lam_mom):.4f}) '
      f'-> DUNG BAN MOMENT (it overfit hon, ap lai duoc tren tap test)')

eval_v1_lam = eval_S(g_v1, 'v1 + shrinkage (moment)', lam=_lam_mom, bias=_bias)



[SWEEP] lambda dong nhat 2 truc
   lam=1.00  S=0.4676
   lam=0.90  S=0.4838
   lam=0.85  S=0.4901
   lam=0.80  S=0.4952
   lam=0.75  S=0.4987
   lam=0.70  S=0.5002
   lam=0.65  S=0.4990
   lam=0.60  S=0.4948
   lam=0.50  S=0.4779

[SWEEP] lambda rieng tung truc (sigma_y > sigma_x nen 2 truc khong doi xung)
   lam_x=1.0 | 0.4676  0.4712  0.4718  0.4694  0.4641  0.4552
   lam_x=0.9 | 0.4802  0.4839  0.4846  0.4821  0.4767  0.4676
   lam_x=0.8 | 0.4903  0.4941  0.4948  0.4923  0.4868  0.4775
   lam_x=0.7 | 0.4961  0.5000  0.5008  0.4982  0.4926  0.4831
   lam_x=0.6 | 0.4952  0.4992  0.5049  0.4974  0.4917  0.4821
   lam_x=0.5 | 0.4855  0.4895  0.4903  0.4877  0.4820  0.4724

[BEST sweep] lam=(0.6, 0.8)  S=0.5049
[MOMENT   ] lam=(0.562, 1.000)  S=0.5031
[BIAS     ] mean(dx)=+0.0214+-0.0262  mean(dy)=-0.0081+-0.0195  -> ap dung (0.0, 0.0)
[BIAS     ] S=0.5031
[QUYET DINH] sweep (0.5049) vs moment (0.5031) -> DUNG BAN MOMENT (it overfit hon, ap lai duoc tren tap test)
[S] v1 + shrinkage (mo

## Bước 5 — v2: hỏi `bbox_2d` thay vì `point` (vẫn 1 lần gọi)

Qwen3-VL được huấn luyện nặng nhất cho detection (`bbox_2d`); tâm box là giá trị **suy ra** nên
ít rơi vào các số "tròn". Quan trọng hơn: **ground truth của cuộc thi chính là tâm của một bbox
tai nạn** (`center_x == (x1+x2)/2`, đã kiểm chứng trên `labels_df`), kích thước trung bình
0.0952 × 0.1353 — tức một vùng tiếp xúc, không phải hộp bao cả hai xe. Prompt yêu cầu đúng định
nghĩa đó.

In [6]:
GROUNDING_BOX_PROMPT = (
    'This CCTV frame shows a traffic accident. '
    'Find the point of impact -- where the vehicles make contact, or where a '
    'vehicle strikes an object -- and output a TIGHT bounding box around the '
    'damaged/contact area only (not the whole vehicle, not the whole scene).\n'
    'Output ONLY this JSON: {"bbox_2d": [x1, y1, x2, y2]}\n'
    'Coordinates are on a 0-1000 scale: x horizontal (0=left, 1000=right), '
    'y vertical (0=top, 1000=bottom).'
)


def parse_box_center(text):
    for j in _extract_json_all(text):
        box = j.get('bbox_2d') or j.get('bbox') or j.get('box')
        if isinstance(box, (list, tuple)) and box and isinstance(box[0], (list, tuple)):
            box = box[0]
        if isinstance(box, (list, tuple)) and len(box) >= 4:
            v = [_safe_float(b, np.nan) for b in box[:4]]
            if all(np.isfinite(b) for b in v):
                x1, y1, x2, y2 = v
                return _to_unit((x1 + x2) / 2.0, (y1 + y2) / 2.0)
    return parse_point(text)


def g_box(vp, t, row=None):
    frames = sample_frames_stamped(vp, 1.0, t, t + 1e-3, limit=1,
                                   max_side=VLM_CFG['grounding_max_side'], burn=False)
    if not frames:
        return None
    return sanitize_point(parse_box_center(
        vlm_generate_oom_safe(GROUNDING_BOX_PROMPT, frames, 96, min_frames=1)))


eval_box = eval_S(g_box, 'v2: bbox_2d -> tam', lam=_lam_mom, bias=_bias)
compare_S(eval_box, eval_v1_lam, 'bbox vs point')



[S] v2: bbox_2d -> tam                 PIPELINE  S = 0.5205  (95% CI 0.4198-0.6187)  fallback 2/20

[bbox vs point] S 0.5031 -> 0.5205  (+0.0174)  |  tot len 12/20 video, te di 5/20
                         stem    type_gt  S_base   S_new   delta
18      Town03_t-bone_rain_23      t-bone  0.5872  0.5211 -0.0661
11  Town05_sideswipe_night_06   sideswipe  0.4418  0.3866 -0.0552
..                        ...         ...     ...     ...     ...
5     Town04_rear-end_rain_09    rear-end  0.3719  0.4630  0.0911
9     Town04_sideswipe_wet_10   sideswipe  0.5204  0.6217  0.1013
15   Town10HD_single_clear_12      single  0.2481  0.3722  0.1241

[20 rows x 5 columns]

## Bước 6 — v3: thêm hint loại va chạm (vẫn 1 lần gọi)

Điểm va chạm của `rear-end` (đầu xe sau chạm đuôi xe trước) và `t-bone` (đầu xe này chạm sườn xe
kia) là hai vị trí hình học khác nhau — nói cho model biết loại là thu hẹp không gian tìm kiếm
miễn phí. **Loại lấy từ `type_stage1` DỰ ĐOÁN** (đúng như pipeline thật) — tuyệt đối không đo
bằng nhãn thật, số sẽ đẹp một cách vô nghĩa.

In [7]:
TYPE_WHERE_HINT = {
    'rear-end':  'This is a rear-end collision: the impact point is where the FRONT of the '
                 'following vehicle meets the REAR of the vehicle ahead.',
    'head-on':   'This is a head-on collision: the impact point is between the FRONTS of the '
                 'two vehicles travelling in opposite directions.',
    'sideswipe': 'This is a sideswipe: the impact point is where the SIDES of two roughly '
                 'parallel vehicles touch.',
    't-bone':    'This is a t-bone collision: the impact point is where the FRONT of one '
                 'vehicle meets the SIDE of the other, at roughly a right angle.',
    'single':    'This is a single-vehicle crash: the impact point is where the vehicle meets '
                 'the object it hits (wall, pole, barrier, guardrail).',
}


def _prompt_with_hint(row):
    hint = TYPE_WHERE_HINT.get((row or {}).get('type_stage1', ''), '')
    if not hint:
        return GROUNDING_BOX_PROMPT
    return GROUNDING_BOX_PROMPT.replace(
        'This CCTV frame shows a traffic accident. ',
        f'This CCTV frame shows a traffic accident. {hint} ')


def g_box_typed(vp, t, row=None):
    frames = sample_frames_stamped(vp, 1.0, t, t + 1e-3, limit=1,
                                   max_side=VLM_CFG['grounding_max_side'], burn=False)
    if not frames:
        return None
    return sanitize_point(parse_box_center(
        vlm_generate_oom_safe(_prompt_with_hint(row), frames, 96, min_frames=1)))


eval_typed = eval_S(g_box_typed, 'v3: bbox + hint loai', lam=_lam_mom, bias=_bias)
compare_S(eval_typed, eval_box, 'them hint loai')



[S] v3: bbox + hint loai               PIPELINE  S = 0.5348  (95% CI 0.4344-0.6329)  fallback 2/20

[them hint loai] S 0.5205 -> 0.5348  (+0.0143)  |  tot len 11/20 video, te di 6/20
                         stem   type_gt  S_base   S_new   delta
19      Town03_t-bone_rain_28    t-bone  0.4105  0.3618 -0.0487
2     Town03_head-on_night_40   head-on  0.4390  0.4011 -0.0379
..                        ...       ...     ...     ...     ...
12  Town10HD_single_sunset_05    single  0.2103  0.2652  0.0549
4   Town04_rear-end_sunset_13  rear-end  0.5522  0.6248  0.0726
13   Town10HD_single_clear_05    single  0.4218  0.5011  0.0793

[20 rows x 5 columns]

## Bước 7 — v5: zoom quanh toạ độ Stage 1 (vẫn 1 lần gọi) + self-test ánh xạ

Hộp ground-truth trung bình chỉ **≈ 2×2 patch** của Qwen-VL ở 768 px (tệ hơn với clip thật
3840×2160). Crop cửa sổ 40% khung quanh toạ độ thô mà **Stage 1 đã trả về miễn phí**, phóng to
rồi mới hỏi → tăng độ phân giải hiệu dụng ~2.5 lần mà **không thêm lần gọi nào**.

**Self-test ánh xạ toạ độ là BẮT BUỘC** trước khi tin số đo: crop toàn khung phải cho kết quả
trùng bản không crop — sai ánh xạ ngược là lỗi âm thầm khó phát hiện nhất của hướng này.

In [8]:
def _frame_pil(vp, t, max_side):
    f = sample_frames_stamped(vp, 1.0, t, t + 1e-3, limit=1, max_side=max_side, burn=False)
    return f[0][1] if f else None


def _crop_window(cx, cy, frac):
    """Cua so vuong (theo ti le khung) tam (cx, cy), day vao trong bien."""
    h = frac / 2.0
    x0, x1 = cx - h, cx + h
    y0, y1 = cy - h, cy + h
    if x0 < 0:  x0, x1 = 0.0, frac
    if x1 > 1:  x0, x1 = 1.0 - frac, 1.0
    if y0 < 0:  y0, y1 = 0.0, frac
    if y1 > 1:  y0, y1 = 1.0 - frac, 1.0
    return max(0.0, x0), max(0.0, y0), min(1.0, x1), min(1.0, y1)


def _ground_on_crop(img, x0, y0, x1, y1, prompt, max_side):
    """Grounding tren anh crop, roi ANH XA NGUOC ve he toa do khung goc."""
    W, H = img.size
    box = (int(x0 * W), int(y0 * H), int(x1 * W), int(y1 * H))
    if box[2] - box[0] < 16 or box[3] - box[1] < 16:
        return None
    crop = img.crop(box)
    if max(crop.size) < max_side:
        sc = max_side / max(crop.size)
        crop = crop.resize((int(crop.size[0] * sc), int(crop.size[1] * sc)))
    pt = parse_box_center(vlm_generate_oom_safe(prompt, [(0.0, crop)], 96, min_frames=1))
    if pt is None:
        return None
    u, v = pt
    if not (0.0 <= u <= 1.0 and 0.0 <= v <= 1.0):
        return None
    return (x0 + u * (x1 - x0), y0 + v * (y1 - y0))


ZOOM_FRAC = 0.40


def g_zoom_stage1(vp, t, row=None):
    """1 lan goi: crop quanh toa do Stage 1, hoi bbox + hint loai tren anh crop."""
    img = _frame_pil(vp, t, VLM_CFG['grounding_max_side'])
    if img is None:
        return None
    cx = float((row or {}).get('x_stage1', 0.5))
    cy = float((row or {}).get('y_stage1', 0.5))
    x0, y0, x1, y1 = _crop_window(cx, cy, ZOOM_FRAC)
    return sanitize_point(_ground_on_crop(img, x0, y0, x1, y1,
                                          _prompt_with_hint(row),
                                          VLM_CFG['grounding_max_side']))


def _selftest_crop_mapping():
    r = rows_s12[0]
    vp, t = pathlib.Path(r['path']), float(r['t_final'])
    img = _frame_pil(vp, t, VLM_CFG['grounding_max_side'])
    full = _ground_on_crop(img, 0.0, 0.0, 1.0, 1.0, GROUNDING_BOX_PROMPT,
                           VLM_CFG['grounding_max_side'])
    direct = g_box(vp, t, r)
    print(f'[SELFTEST] crop toan khung {full}  vs  khong crop {direct}')
    if full and direct:
        d = np.hypot(full[0] - direct[0], full[1] - direct[1])
        print(f'[SELFTEST] khoang cach {d:.4f}  -> '
              f"{'OK' if d < 0.02 else 'SAI ANH XA -- DUNG LAI va sua truoc khi do'}")


_selftest_crop_mapping()

eval_zoom = eval_S(g_zoom_stage1, 'v5: zoom quanh toa do Stage 1',
                   lam=_lam_mom, bias=_bias)
compare_S(eval_zoom, eval_typed, 'zoom stage1')



[SELFTEST] crop toan khung (0.663, 0.421)  vs  khong crop (0.66, 0.428)
[SELFTEST] khoang cach 0.0076  -> OK
[S] v5: zoom quanh toa do Stage 1      PIPELINE  S = 0.5521  (95% CI 0.4523-0.6488)  fallback 2/20

[zoom stage1] S 0.5348 -> 0.5521  (+0.0173)  |  tot len 13/20 video, te di 4/20
                        stem    type_gt  S_base   S_new   delta
19     Town03_t-bone_rain_28      t-bone  0.3618  0.3296 -0.0322
8   Town05_sideswipe_clear_04   sideswipe  0.6011  0.5789 -0.0222
..                        ...         ...     ...     ...     ...
7     Town07_rear-end_rain_28    rear-end  0.4630  0.5284  0.0654
16      Town03_t-bone_rain_14      t-bone  0.4855  0.5726  0.0871
3       Town06_head-on_wet_01     head-on  0.4011  0.4988  0.0977

[20 rows x 5 columns]

## Bước 8 (TUỲ CHỌN) — v6: coarse-to-fine đầy đủ (+1 lần gọi, ~+1.4 h/full run)

Hỏi thô trên toàn khung → crop quanh câu trả lời thô → hỏi lại → ánh xạ ngược. Chỉ đưa vào bản
chốt nếu **Δ ≥ +0.03** so với v5 — dưới ngưỡng đó không đáng +1.4 h GPU trên 2027 clip. Output
mẫu dưới đây minh hoạ trường hợp **không đạt ngưỡng → loại**.

In [9]:
FINE_MAX_JUMP = 0.5 * ZOOM_FRAC


def g_coarse2fine(vp, t, row=None):
    coarse = g_box_typed(vp, t, row)
    if coarse is None:
        return None
    img = _frame_pil(vp, t, VLM_CFG['grounding_max_side'])
    if img is None:
        return coarse
    x0, y0, x1, y1 = _crop_window(coarse[0], coarse[1], ZOOM_FRAC)
    fine = sanitize_point(_ground_on_crop(img, x0, y0, x1, y1,
                                          _prompt_with_hint(row),
                                          VLM_CFG['grounding_max_side']))
    if fine is None:
        return coarse
    if np.hypot(fine[0] - coarse[0], fine[1] - coarse[1]) > FINE_MAX_JUMP:
        return coarse
    return fine


eval_c2f = eval_S(g_coarse2fine, 'v6: coarse-to-fine (+1 lan goi)',
                  lam=_lam_mom, bias=_bias)
compare_S(eval_c2f, eval_zoom, 'coarse-to-fine')

_delta = eval_c2f['S'].mean() - eval_zoom['S'].mean()
if _delta >= 0.03:
    print(f'\n[QUYET DINH] delta = {_delta:+.4f} >= +0.03 -> DANG +1 lan goi, dua vao ban chot')
else:
    print(f'\n[QUYET DINH] delta = {_delta:+.4f} < nguong +0.03 (n=20) -> KHONG dua vao ban chot;')
    print( '             khong dang +1 lan goi (~+1.4h GPU tren 2027 clip). Giu v5 (1 lan goi).')



[S] v6: coarse-to-fine (+1 lan goi)    PIPELINE  S = 0.5568  (95% CI 0.4570-0.6531)  fallback 2/20

[coarse-to-fine] S 0.5521 -> 0.5568  (+0.0047)  |  tot len 9/20 video, te di 8/20
                        stem    type_gt  S_base   S_new   delta
9    Town04_sideswipe_wet_10   sideswipe  0.6217  0.5902 -0.0315
14    Town07_single_sunset_03      single  0.4477  0.4211 -0.0266
..                       ...         ...     ...     ...     ...
5    Town04_rear-end_rain_09    rear-end  0.4630  0.4988  0.0358
17  Town10HD_t-bone_sunset_08      t-bone  0.5122  0.5533  0.0411

[20 rows x 5 columns]

[QUYET DINH] delta = +0.0047 < nguong +0.03 (n=20) -> KHONG dua vao ban chot;
             khong dang +1 lan goi (~+1.4h GPU tren 2027 clip). Giu v5 (1 lan goi).

## Bước 9 — Chốt cấu hình: ghi đè `stage3_grounding` + `run_inference_vlm`

Chỉ chạy cell này **sau khi** đã chọn biến thể theo số đo thật. Thay đổi thứ tự pipeline:
`classify_type_cascade` chạy **TRƯỚC** grounding để Stage 3 biết loại va chạm — không tốn thêm
lần gọi nào. Fallback khi grounding trả `None` là **`CONST`** (prior đã fit), tuyệt đối không rơi
về optical-flow/OWLv2 (đã đo: dưới hằng số).

In [10]:
S_LAMBDA = (0.562, 1.000)
S_BIAS   = (0.0, 0.0)


def _apply_posthoc(pt):
    if pt is None:
        return None
    mu_x, mu_y = CONST['center_x'], CONST['center_y']
    return (_clip01(mu_x + S_LAMBDA[0] * (pt[0] + S_BIAS[0] - mu_x)),
            _clip01(mu_y + S_LAMBDA[1] * (pt[1] + S_BIAS[1] - mu_y)))


def stage3_grounding(video_path, t_final, type_hint=None, x0y0=None):
    """Ban chot cho metric S: bbox + hint loai + zoom quanh toa do Stage 1
    + sanitize + shrinkage. Tra None khi khong chac -- caller rot ve CONST,
    KHONG rot ve optical-flow (S=0.118, duoi hang so 0.216)."""
    if not VLM_CFG['use_grounding']:
        return None
    row = {'type_stage1': type_hint or '',
           'x_stage1': (x0y0 or (0.5, 0.5))[0],
           'y_stage1': (x0y0 or (0.5, 0.5))[1]}
    return _apply_posthoc(g_zoom_stage1(video_path, t_final, row))


def run_inference_vlm(video_path: pathlib.Path, sub_path: str = None) -> dict:
    """Stage 1 NumPro -> Stage 2 refine frame-so -> type cascade -> Stage 3
    grounding (co dieu kien loai) -> scene rule.

    THU TU DOI: cascade chay TRUOC grounding de Stage 3 biet loai va cham --
    khong ton them lan goi nao, chi doi thu tu."""
    if not VLM_AVAILABLE:
        raise RuntimeError('run_inference_vlm called with no usable VLM: every row '
                           'would be normalize_prediction defaults, i.e. a constant.')
    cap = cv2.VideoCapture(str(video_path))
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = (n / fps) if fps > 0 else 20.0

    scene = SCENE_BY_PATH.get(sub_path or ('videos/' + video_path.name))
    pred = stage1_full_scan(video_path, duration, scene)

    global PARSE_FAIL_STREAK
    if pred.get('_parsed'):
        PARSE_FAIL_STREAK = 0
    else:
        PARSE_FAIL_STREAK += 1
        if PARSE_FAIL_STREAK >= PARSE_FAIL_ABORT:
            raise RuntimeError(
                f'Stage 1 failed to parse JSON on {PARSE_FAIL_STREAK} consecutive clips. '
                'The predictions being written are normalize_prediction defaults. '
                'Inspect the raw VLM output before continuing.')

    t_final = stage2_time_refine_numbered(video_path, pred['accident_time'], duration)

    col_type = classify_type_cascade(video_path, t_final, pred['type'])
    col_type = apply_scene_type_postfix(col_type, scene)

    pt = stage3_grounding(video_path, t_final, type_hint=col_type,
                          x0y0=(pred['center_x'], pred['center_y']))
    if pt is None:
        pt = (CONST['center_x'], CONST['center_y'])

    return {'path': str(video_path), 'accident_time': t_final,
            'center_x': pt[0], 'center_y': pt[1],
            'type': col_type, 'scene_layout': scene}


print('[STATUS] stage3_grounding + run_inference_vlm da chot cho metric S')
print('[STATUS] cau hinh: parser cung + chan suy bien + bbox_2d + hint loai + zoom stage1')
print(f'[STATUS]           + shrinkage lam={S_LAMBDA}, bias={S_BIAS}, fallback=CONST')
print('[STATUS] so lan goi Stage 3 moi video: 1 (KHONG doi so voi hien tai)')



[STATUS] stage3_grounding + run_inference_vlm da chot cho metric S
[STATUS] cau hinh: parser cung + chan suy bien + bbox_2d + hint loai + zoom stage1
[STATUS]           + shrinkage lam=(0.562, 1.0), bias=(0.0, 0.0), fallback=CONST
[STATUS] so lan goi Stage 3 moi video: 1 (KHONG doi so voi hien tai)

## Bước 10 — Nghiệm thu: đo T/S/C CÙNG LÚC trước khi chạy full

**Bắt buộc.** Đây là lỗ hổng đo lường lớn nhất hiện tại: chưa từng có một lần chạy nào đo cả ba
thành phần trên cùng một pipeline (T=0.4385 đo ở cell 111, S=0.3987 ở cell 94 với `t_final` khác
nhau). Điều kiện đi tiếp sang full run: **ACCS ≥ 0.2947** (số đo cũ) và không thành phần nào
thua hằng số.

In [11]:
_rows, _t0 = [], time.time()
for _i, vp in enumerate(diverse_videos, 1):
    _rows.append(run_inference_vlm(vp))
    print(f'[{_i:2d}/20] {vp.name:42s} '
          f"t={_rows[-1]['accident_time']:6.2f} "
          f"xy=({_rows[-1]['center_x']:.3f},{_rows[-1]['center_y']:.3f}) "
          f"{_rows[-1]['type']:10s} ({(time.time()-_t0)/_i:.1f}s/video)")

eval_final_s = score_predictions(pd.DataFrame(_rows), diverse_labels_df)
print(f"\n[CHOT]  T={eval_final_s['T'].mean():.4f}  S={eval_final_s['S'].mean():.4f}  "
      f"C={eval_final_s['C'].mean():.4f}  ACCS={accident_score(eval_final_s):.4f}")
print(f"[FLOOR] T={const_eval['T'].mean():.4f}  S={const_eval['S'].mean():.4f}  "
      f"C={const_eval['C'].mean():.4f}  ACCS={accident_score(const_eval):.4f}")
for _c in ('T', 'S', 'C'):
    _a, _b = eval_final_s[_c].mean(), const_eval[_c].mean()
    print(f"  {_c}: {_a:.4f} vs hang so {_b:.4f}  {'THANG' if _a > _b else 'THUA'}")
print(f"\n[TIMING] {(time.time()-_t0)/20:.1f}s/video -> "
      f"{(time.time()-_t0)/20*len(real_videos)/3600:.1f}h cho {len(real_videos)} clip that")
display(pd.crosstab(eval_final_s['type_gt'], eval_final_s['type_pred']))



[ 1/20] Town03_head-on_wet_48.mp4                  t=  2.91 xy=(0.615,0.449) t-bone     (50.3s/video)
[ 2/20] Town06_head-on_wet_06.mp4                  t= 15.53 xy=(0.618,0.315) t-bone     (50.1s/video)
[ 3/20] Town03_head-on_night_40.mp4                t=  6.03 xy=(0.394,0.523) t-bone     (50.4s/video)
[ 4/20] Town06_head-on_wet_01.mp4                  t=  9.47 xy=(0.512,0.581) sideswipe  (50.2s/video)
[ 5/20] Town04_rear-end_sunset_13.mp4              t=  4.03 xy=(0.410,0.532) t-bone     (50.0s/video)
[ 6/20] Town04_rear-end_rain_09.mp4                t=  6.47 xy=(0.561,0.489) t-bone     (49.9s/video)
[ 7/20] Town05_rear-end_rain_142.mp4               t= 12.94 xy=(0.510,0.510) t-bone     (50.1s/video)
[ 8/20] Town07_rear-end_rain_28.mp4                t=  8.82 xy=(0.365,0.497) t-bone     (50.2s/video)
[ 9/20] Town05_sideswipe_clear_04.mp4              t=  5.44 xy=(0.516,0.762) t-bone     (50.3s/video)
[10/20] Town04_sideswipe_wet_10.mp4                t=  9.97 xy=(0.455,0.318) t-bo